In [ ]:
# --- SETUP (same in every notebook) -----------------------------------
OPERATOR = "nikolai"                       # <- your name (your own working copy of the code)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
ROOT = "/content/drive/MyDrive/pei_landuse"
REPO = f"{ROOT}/code/{OPERATOR}/pei-landuse"
assert os.path.exists(REPO), "No code yet: run notebooks/00_git.ipynb (clone) first."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{REPO}/src"); os.chdir(REPO)
%load_ext autoreload
%autoreload 2
print(subprocess.run("git branch --show-current; git log -1 --oneline; git status --short | head", shell=True, capture_output=True, text=True).stdout)

In [ ]:
CONFIG = "configs/infer_2020_prince.yaml"

import torch
from pei.config import load_config, make_run_dir
from pei.infer import run_inference, load_raw_detections, resolve_raster
from pei.merge import merge_detections
from pei.export import finalize

cfg = load_config(CONFIG)
run_dir = make_run_dir(cfg)
raster = resolve_raster(cfg, run_dir)
print("run folder:", run_dir)

In [ ]:
if not cfg["paths"].get("raw_detections"):
    from pei.model import load_for_inference
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_for_inference(cfg, device)
    run_inference(cfg, model, run_dir, device, raster_path=raster)

In [ ]:
raw = load_raw_detections(run_dir, cfg["paths"].get("raw_detections"))
fields = merge_detections(cfg, raw, run_dir, raster)

In [ ]:
out = finalize(cfg, fields, run_dir)
print(out)